In [1]:
from platform import python_version
print(python_version())

3.11.14


In [2]:
import os, sys, yaml
from pathlib import Path
from dotenv import load_dotenv

import numpy as np
import pandas as pd
pd.set_option('display.width', 100)
pd.set_option('max_colwidth', 80)
pd.set_option("display.precision", 3)

import seaborn as sns
sns.set_context("notebook", font_scale=1.4)

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
%matplotlib inline

sys.path.insert(1, '../src/')

ROOT0 = Path("/home/flavio/uv/perturb_agent/")
ROOT_SRC = ROOT0 / "src"
ROOT_COLAB = ROOT0 / "colab"

if str(ROOT_SRC) not in sys.path:
    sys.path.append(str(ROOT_SRC))

print("ROOT0:", ROOT0)
print("ROOT_SRC added:", ROOT_SRC)

from libs.Basic import *
from libs.MTD_lib import *
from libs.dashcyto_lib import DASH_CYTO
from libs.calc_degs_lib import CALC_DEGS

from IPython.display import display, HTML
# display(HTML("<style>.container { width:100% !important; }</style>"))
display(HTML("<style>:root { --jp-notebook-max-width: 100% !important; }</style>"))

with open('../params.yml', 'r') as file:
    dic_yml = yaml.safe_load(file)

# print(dic_yml)

ROOT0: /home/flavio/uv/perturb_agent
ROOT_SRC added: /home/flavio/uv/perturb_agent/src


/home/flavio/uv/perturb_agent/.venv/lib/python3.11/site-packages/Bio/__init__.py:138: BiopythonWarning: You may be importing Biopython from inside the source tree. This is bad practice and might lead to downstream issues. In particular, you might encounter ImportErrors due to missing compiled C extensions. We recommend that you try running your code from outside the source tree. If you are outside the source tree then you have a pyproject.toml file in an unexpected directory: /home/flavio/uv/perturb_agent/.venv/lib/python3.11/site-packages
  warnings.warn(


In [3]:
root0 = Path(dic_yml['root0'])
root0_data = Path(dic_yml['root0_data'])
root_colab = root0 / 'colab'

email = os.getenv('email')

i_project=0

project_list = dic_yml['project_list']
n = len(project_list)
project = project_list[i_project]

s_project_list = dic_yml['s_project_list']
s_project = s_project_list[i_project]
assert n==len(project_list), f"Error project_list: there are {n} projects"


root_project = root0_data / s_project
PSI_ID = 'TCGA-BRCA'
PSI_ID = 'TCGA-ACC'
PSI_ID = 'TCGA-CESC'

disease = PSI_ID

root_project = create_dir(root0_data, s_project)
root_disease = create_dir(root_project, PSI_ID)

CONTEXT_DISESE = 'xxxx'
context_disease = CONTEXT_DISESE

gene_protein = dic_yml['gene_protein']
s_omics = dic_yml['s_omics']

has_age = dic_yml['has_age']
has_gender = dic_yml['has_gender']

exp_normalization = dic_yml['exp_normalization']
normalization = 'quantile_norm' if exp_normalization == True else 'not_normalized'

LFC_cut_inf = dic_yml['LFC_cut_inf']
s_pathw_enrichm_method = dic_yml['s_pathw_enrichm_method']
ptw_min_num_of_degs_cut = dic_yml['ptw_min_num_of_degs_cut']

tolerance_pPMI = dic_yml['tolerance_pPMI']
type_sat_ptw_index = dic_yml['type_sat_ptw_index']
saturation_lfc_param = dic_yml['saturation_lfc_param']

pval_pathway_cutoff = dic_yml['pval_pathway_cutoff']
fdr_pathway_cutoff = dic_yml['fdr_pathway_cutoff']
num_of_genes_cutoff = dic_yml['num_of_genes_cutoff']
enr_db_list = dic_yml['enr_db_list']

case_list = dic_yml['case_list']
dic_case_list = dic_yml['dic_case_list']

std_filename      = dic_yml['std_filename']
std_filename_list = dic_yml['std_filename_list']

min_lfc_modulation = dic_yml['min_lfc_modulation']
num_of_genes_list  = dic_yml['num_of_genes_list']
pPMI_normalized  = dic_yml['pPMI_normalized']

#--- max len for formatting purposes
s_len_case  = dic_yml['s_len_case']

n_sentences = dic_yml['n_sentences']
run_list = dic_yml['run_list']
chosen_model_list = dic_yml['chosen_model_list']
i_dfp_list = dic_yml['i_dfp_list']
chosen_model_sampling = dic_yml['chosen_model_sampling']

fdr_ptw_cutoff_list = np.arange(0.05, 0.80, 0.05)
lfc_list = np.round(np.arange(1.0, -0.01, -.025), 3)
fdr_list = np.arange(0.05, 0.76, .01)

cfg = Config(root0=root0, root_disease=root_disease, disease=disease, case_list=case_list)
case = case_list[0]

n_genes_annot_ptw, n_degs, n_degs_in_ptw, n_degs_not_in_ptw, degs_in_all_ratio = -1,-1,-1,-1,-1

LFC_cut, lfc_FDR_cut, n_degs, n_degs_up, n_degs_dw = cfg.get_best_lfc_cutoff(case, 'not_normalized')

print(f"project '{project}', s_project '{s_project}'")
print(f"G/P LFC cutoffs: lfc={LFC_cut:.3f}; fdr={lfc_FDR_cut:.3f} - LFC_cut_inf={LFC_cut_inf:.3f}")
print(f"Pathway cutoffs: pval={pval_pathway_cutoff:.3f}; fdr={fdr_pathway_cutoff:.3f}; num of genes={num_of_genes_cutoff}")

project 'TCGA', s_project 'TCGA'
G/P LFC cutoffs: lfc=1.000; fdr=0.050 - LFC_cut_inf=0.400
Pathway cutoffs: pval=0.050; fdr=0.050; num of genes=3


In [4]:
mtd = MTD(disease=disease, gene_protein=gene_protein, s_omics=s_omics, project=project, s_project=s_project, root0=root0, root0_data=root0_data,
          case_list=case_list, dic_case_list=dic_case_list, has_age=has_age, has_gender=has_gender, exp_normalization=exp_normalization,
          std_filename=std_filename, std_filename_list=std_filename_list,
          geneset_num=0, ptw_min_num_of_degs_cut=ptw_min_num_of_degs_cut,
          tolerance_pPMI=tolerance_pPMI, s_pathw_enrichm_method=s_pathw_enrichm_method,
          LFC_cut_inf=LFC_cut_inf, fdr_ptw_cutoff_list=fdr_ptw_cutoff_list,
          num_of_genes_list=num_of_genes_list, lfc_list=lfc_list, fdr_list=fdr_list, 
          min_lfc_modulation=min_lfc_modulation, type_sat_ptw_index=type_sat_ptw_index,
          saturation_lfc_param=saturation_lfc_param, enr_db_list=enr_db_list, pPMI_normalized=pPMI_normalized)

print(">>> Roots", mtd.root0, mtd.root_disease)
case = case_list[0]
print(">>>", case)

mtd.cfg.set_default_best_lfc_cutoff(mtd.normalization, LFC_cut=1, lfc_FDR_cut=0.05)
ret, degs, degs_ensembl, dfdegs = mtd.open_case(case, prompt_verbose=True, verbose=False)
print("\nEcho Parameters:")
print(mtd.echo_parameters())

Start opening tables ....
Building synonym dictionary ...

>>> Roots /home/flavio/uv/perturb_agent /home/flavio/uv/perturb_agent/data/TCGA/TCGA-CESC
>>> Tumor
>>> case Tumor
	DEGs 20006
		Up (#10358)
		Dw (#9648)

Up-regulated per biotype
                               biotype     n
0                            IG_C_gene    12
1                      IG_C_pseudogene     4
2                            IG_D_gene     1
3                            IG_J_gene     9
4                      IG_J_pseudogene     1
5                            IG_V_gene   124
6                      IG_V_pseudogene    50
7                              Mt_tRNA    17
8                                  TEC   112
9                            TR_C_gene     5
10                           TR_D_gene     1
11                           TR_J_gene    10
12                           TR_V_gene    69
13                     TR_V_pseudogene     1
14                              lncRNA  3193
15                               miRNA   

In [5]:
mtd.geneset_num, mtd.geneset_lib

(0, 'Reactome_Pathways_2024')

In [6]:
mtd.gene.df_my_gene.head(2)

,entrezid,symbol,geneid,name,synonyms,other_names,ec_enzyme,refseq_gen,refseq_prot,refseq_rna,...,acessions_rna,acessions_trans,map_location,dic_panther,ortholog,dic_uniprot,swissprot,wikipedia,refseq_gen,refseq_rna
0,ENSG00000121410,A1BG,1,alpha-1-B glycoprotein,"['A1B', 'ABG', 'GAB', 'HYST2477']","['HEL-S-163pA', 'alpha-1B-glycoprotein', 'epididymis secretory sperm binding...",NaN,NaN,NP_570602.2,NaN,...,"['AA484435.1', 'AB073611.1', 'AF414429.1', 'AI022193.1', 'AK055885.1', 'AK05...","[{'protein': 'AAH35719.1', 'rna': 'BC035719.1'}, {'protein': 'BAG51591.1', '...",19q13.43,"{'HGNC': '5', '_license': 'http://pantherdb.org/tou.jsp', 'ortholog': [{'MGI...","[{'MGI': '2152878', 'ortholog_type': 'LDO', 'panther_family': 'PTHR11738', '...","{'Swiss-Prot': 'P04217', 'TrEMBL': ['M0R009', 'V9HWD8']}",P04217,{'url_stub': 'A1BG (gene)'},"['NC_000019.10', 'NC_060943.1']",NM_130786.4
1,ENSG00000268895,A1BG-AS1,503538,A1BG antisense RNA 1,"['A1BG-AS', 'A1BGAS', 'NCRNA00181']","['A1BG antisense RNA (non-protein coding)', 'A1BG antisense RNA 1 (non-prote...",NaN,NaN,NaN,NaN,...,"['AK027222.1', 'BC006144.1', 'BC040926.1', 'NR_015380.2']",NaN,19q13.43,NaN,NaN,NaN,NaN,NaN,"['NC_000019.10', 'NC_060943.1']",NR_015380.2


### Studying default non-conding genes

In [7]:
ret, degs, degs_ensembl, dfdegs = mtd.open_case_params(case, LFC_cut=1, lfc_FDR_cut=0.05, verbose=False)
print("\nEcho Parameters - default:")
print(mtd.echo_parameters())



Echo Parameters - default:
	10993/10993 DEGs/ensembl.
		Up 5838/5838 DEGs/ensembl.
		Dw 5155/5155 DEGs/ensembl.

Found 0 (best=0) pathways for geneset num=0 'Reactome_Pathways_2024'
Pathway cutoffs p-value=0.050 fdr=0.050 min genes=3No enrichment analysis was calculated.


In [8]:
dflfc = mtd.dflfc
dflfc.biotype.unique()

array(['lncRNA', 'protein_coding', 'transcribed_unitary_pseudogene',
       'TEC', 'unprocessed_pseudogene', 'Mt_tRNA',
       'transcribed_unprocessed_pseudogene', 'processed_pseudogene',
       'snoRNA', 'IG_V_gene', 'miRNA', 'IG_C_gene',
       'polymorphic_pseudogene', 'transcribed_processed_pseudogene',
       'IG_C_pseudogene', 'snRNA', 'misc_RNA', 'IG_V_pseudogene',
       'TR_V_gene', 'IG_J_pseudogene', 'IG_J_gene', 'TR_D_gene',
       'TR_C_gene', 'TR_J_gene', 'unitary_pseudogene', 'rRNA_pseudogene',
       'translated_unprocessed_pseudogene'], dtype=object)

### dflfc_noncod

### Calculating DEGs statistics

### For each LFC/FDR cutoff set we get diferent set of DEGs
  - LFC: LFC cutoff and FDR_LFC cutoff
  - Pathway: fdr and pval pathway cutoff and min num of genes

### Up and Down
  - Up and Down DEGs/DAPs
  - Up and Down in pathways

### there are 2 statistical tables
  - pval/fdr cutoff x degs
  - pval/fdr/geneset/quantile degs_in_pathway, num_pathways

### Biotypes

https://www.ensembl.org/info/genome/genebuild/biotypes.html

### Ensembl Gallery

https://www.ensembl.org/info/website/gallery.html

In [11]:
lfc_cut = 1.0
fdr_cut = 0.05

verbose=False
force=True

dfnc, msg = mtd.filter_non_coding(lfc_cut=lfc_cut, fdr_cut=fdr_cut, verbose=verbose, force=force)
print(msg, '\n')
dfnc.head(6)

len noncoding: 4711/32213, unique biotypes: ['IG_V_pseudogene' 'lncRNA' 'miRNA' 'misc_RNA' 'polymorphic_pseudogene'
 'processed_pseudogene' 'rRNA_pseudogene' 'snRNA' 'snoRNA'
 'transcribed_processed_pseudogene' 'transcribed_unitary_pseudogene'
 'transcribed_unprocessed_pseudogene' 'translated_unprocessed_pseudogene'
 'unitary_pseudogene' 'unprocessed_pseudogene'] 



,ensembl_id,symbol,biotype,lfc,lfcSE,statistic,pval,fdr,baseMean,method,abs_lfc
0,ENSG00000254174,IGHV1-12,IG_V_pseudogene,4.321,1.666,2.594,9.491e-03,0.029,3.086,DESeq2,4.321
3,ENSG00000253274,IGHV1-67,IG_V_pseudogene,5.232,1.806,2.898,3.757e-03,0.013,5.779,DESeq2,5.232
5,ENSG00000259337,IGHV1OR15-2,IG_V_pseudogene,5.356,1.537,3.485,4.918e-04,0.002,12.416,DESeq2,5.356
6,ENSG00000261704,IGHV1OR16-1,IG_V_pseudogene,4.689,1.889,2.482,1.306e-02,0.037,1.991,DESeq2,4.689
9,ENSG00000278473,IGHV3-41,IG_V_pseudogene,4.129,1.666,2.479,1.319e-02,0.038,2.610,DESeq2,4.129
10,ENSG00000229092,IGHV3-47,IG_V_pseudogene,4.256,1.451,2.932,3.368e-03,0.012,1.456,DESeq2,4.256


In [10]:
biotype_list = ['lncRNA', 'transcribed_unitary_pseudogene',
       'unprocessed_pseudogene', 
       'transcribed_unprocessed_pseudogene', 'processed_pseudogene',
       'snoRNA', 'miRNA',
       'polymorphic_pseudogene', 'transcribed_processed_pseudogene',
       'snRNA', 'misc_RNA', 'IG_V_pseudogene',
       'unitary_pseudogene', 'rRNA_pseudogene',
       'translated_unprocessed_pseudogene']


dflfc_noncod = mtd.dflfc_ori[ mtd.dflfc_ori.biotype.isin(biotype_list)].copy()
dflfc_noncod = dflfc_noncod.sort_values(["biotype", "symbol"])
dflfc_noncod.reset_index(drop=True, inplace=True)

uniq_biotypes = np.unique(dflfc_noncod.biotype)
print(f"len noncoding: {len(dflfc_noncod)}/{len(mtd.dflfc_ori)}, unique biotypes: {uniq_biotypes}")

# fname = f'non_codings_for_{mtd.disease}.tsv'
# pdwritecsv(dflfc_noncod, fname, mtd.root_nc, verbose=True)
dflfc_noncod.head()

len noncoding: 13731/32213, unique biotypes: ['IG_V_pseudogene' 'lncRNA' 'miRNA' 'misc_RNA' 'polymorphic_pseudogene'
 'processed_pseudogene' 'rRNA_pseudogene' 'snRNA' 'snoRNA'
 'transcribed_processed_pseudogene' 'transcribed_unitary_pseudogene'
 'transcribed_unprocessed_pseudogene' 'translated_unprocessed_pseudogene'
 'unitary_pseudogene' 'unprocessed_pseudogene']


,ensembl_id,symbol,biotype,lfc,lfcSE,statistic,pval,fdr,baseMean,method,abs_lfc
0,ENSG00000254174,IGHV1-12,IG_V_pseudogene,4.321,1.666,2.594,0.009,0.029,3.086,DESeq2,4.321
1,ENSG00000253709,IGHV1-14,IG_V_pseudogene,2.450,1.751,1.399,0.162,0.274,0.830,DESeq2,2.450
2,ENSG00000254046,IGHV1-17,IG_V_pseudogene,1.528,1.889,0.809,0.419,0.554,0.941,DESeq2,1.528
3,ENSG00000253274,IGHV1-67,IG_V_pseudogene,5.232,1.806,2.898,0.004,0.013,5.779,DESeq2,5.232
4,ENSG00000253703,IGHV1-68,IG_V_pseudogene,3.450,2.814,1.226,0.220,0.345,0.824,DESeq2,3.450


In [ ]:
cols = ['LNC', 'protein_coding', 'hasSymbol', 'UNK']

dflfc_noncod = dflfc[ ~dflfc.biotype.isin(cols)]
print(len(dflfc_noncod), dflfc_noncod.biotype.unique())

fname = f'non_codings_degs_for_{mtd.case}.tsv'
# pdwritecsv(dflfc_noncod, fname, mtd.root_result, verbose=True)

dflfc_noncod.head()

### LncRNA_Disease v3.0

http://www.rnanut.net/lncrnadisease/index.php/home/

http://www.rnanut.net/lncrnadisease/

### miR2Disease

http://watson.compbio.iupui.edu:8080/miR2Disease/index.jsp

### Disease-associated Enhancer Catalog

http://biocc.hrbmu.edu.cn/DiseaseEnhancer/

In [ ]:
root_lrd = os.path.join(root_colab, 'lncRNA')
fname = 'all_ncRNA_disease_information.tsv'

fullname = os.path.join(root_lrd, fname)

if os.path.exists(fullname):
    dfdis = pdreadcsv(fname, root_lrd)
    print(len(dfdis))
else:
    dfdis = pd.DataFrame()
    
dfdis.head(3)

In [ ]:
symbols = dflfc_noncod.symbol
print("dflfc_noncod", len(symbols))
"; ".join(symbols)

### manually from genecards

TPTE2P1 (TPTE2 Pseudogene 1) is a Pseudogene. Diseases associated with TPTE2P1 include Hepatocellular Carcinoma and Colorectal Cancer.

PRSS30P (Serine Protease 30, Pseudogene) is a Pseudogene

LINC00472 (Long Intergenic Non-Protein Coding RNA 472) - Diseases associated with LINC00472 include Lung Cancer Susceptibility 3 and Squamous Cell Carcinoma, Head and Neck.

ZNF702P (Zinc Finger Protein 702, Pseudogene) - 

### maayanlab.cloud/Harmonizome/gene

https://maayanlab.cloud/Harmonizome/gene

LOC100190940 - has 673 functional associations with biological entities spanning 3 categories (molecular profile, cell line, cell type or tissue, gene, protein or microRNA) extracted from 12 datasets. https://maayanlab.cloud/Harmonizome/gene/LOC100190940


LOC440896 - has 390 functional associations with biological entities spanning 7 categories (molecular profile, organism, disease, phenotype or trait, chemical, functional term, phrase or reference, cell line, cell type or tissue, gene, protein or microRNA) extracted from 19 datasets. https://maayanlab.cloud/Harmonizome/gene/LOC440896


LOC149351 - has 344 functional associations with biological entities spanning 6 categories (organism, disease, phenotype or trait, chemical, functional term, phrase or reference, cell line, cell type or tissue, gene, protein or microRNA) extracted from 10 datasets. https://maayanlab.cloud/Harmonizome/gene/LOC149351

LOC644662 - has 252 functional associations with biological entities spanning 5 categories (molecular profile, disease, phenotype or trait, functional term, phrase or reference, cell line, cell type or tissue, gene, protein or microRNA) extracted from 9 datasets.

LOC643923 - has 1,208 functional associations with biological entities spanning 4 categories (molecular profile, disease, phenotype or trait, cell line, cell type or tissue, gene, protein or microRNA) extracted from 17 datasets.

In [ ]:
symbols_out = [x for x in symbols if x not in dfdis['ncRNA Symbol'].to_list()]
print(len(symbols_out))
"; ".join(symbols_out)

In [ ]:
dfin = dfdis[dfdis['ncRNA Symbol'].isin(symbols)]
print("dflfc_noncod symbols", len(symbols))
print("dfdis", len(dfdis))
print("df commons", len(dfin))
print("df commons unique", len(dfin['ncRNA Symbol'].unique()))
dfin

In [ ]:
symb_identified_list = dfin['ncRNA Symbol'].unique()
symb_identified_list

## Referências para estes LNC do WNT

### LINC00472

17. Seo D., Roh J., Chae Y., Kim W. Gene expression profiling after LINC00472 overexpression in an NSCLC cell line1. Cancer Biomark. Sect. A Dis. Markers. 2021;32:175–188. doi: 10.3233/CBM-210242. [PubMed] [CrossRef] [Google Scholar]

  -  Bladder carcinoma	Growth/metastasis
93. Li L, Qi F, Wang K. Matrine restrains cell growth and metastasis by up-regulating LINC00472 in bladder carcinoma. Cancer Manage Res (2020) 12:1241–51.   10.2147/cmar.S224701 [PMC free article] [PubMed] [CrossRef] [Google Scholar]

### TPTE2P1

63. Corrigendum. Downregulation of TPTE2P1 inhibits migration and invasion of gallbladder cancer cells. Chem Biol Drug Des. 2018;92:1816. [PubMed] [Google Scholar]




In [ ]:
mtd.root_nc

In [ ]:
fname = f"lnc_disease_identification_for_{mtd.case}.tsv"
pdwritecsv(dfin, fname, mtd.root_nc)

In [ ]:
print(len(dfin))
dfin[ dfin['ncRNA Symbol'] == symb_identified_list[0] ]

In [ ]:
dfin[ dfin['ncRNA Symbol'] == symb_identified_list[1] ]

In [ ]:
dflfc_LFC = dflfc[ dflfc.biotype == 'LNC']

fname = f'LNC_special_non_codings_degs_for_{mtd.case}.tsv'
# pdwritecsv(dflfc_LFC, fname, mtd.root_result, verbose=True)

print(len(dflfc_LFC))
dflfc_LFC.head()

In [ ]:
symbols = dflfc_LFC.symbol.unique()
len(symbols), ", ".join(symbols)

In [ ]:
df_ident_LNC = dfdis[dfdis['ncRNA Symbol'].isin(symbols)]
symb_identified_list = df_ident_LNC['ncRNA Symbol'].unique()
symb_identified_list

### G4

In [ ]:
case = case_list[1]

ret, degs, degs_ensembl, dfdegs = mtd.open_case_params(case, LFC_cut=1, lfc_FDR_cut=0.05, verbose=False)
print("\nEcho Parameters - default:")
mtd.echo_parameters()

In [ ]:
dflfc = mtd.dflfc
dflfc.biotype.unique()

In [ ]:
biotype_list = ['LNC']

dflfc_noncod = dflfc[ dflfc.biotype.isin(biotype_list)]
print(len(dflfc_noncod), dflfc_noncod.biotype.unique())

fname = f'non_codings_degs_for_{mtd.case}.tsv'
# pdwritecsv(dflfc_noncod, fname, mtd.root_result, verbose=True)

cols = ['probe', 'symbol', 'symbol_prev', 'symb_or_syn', 'biotype', '_type', 'lfc', 'abs_lfc',
        'pval', 'fdr', 'mean_exp', 't', 'B', 'description', 'desc_gff', 'description_prev',
        'accession', 'ensembl_id', 'ensembl_transc_id', 'geneid', 'cytoband', 'symbol_pipe',
        'seqname', 'start', 'end', 'go_id', 'seq']

cols = ['probe', 'symbol', 'accession', 'geneid', 'ensembl_id', 'biotype', '_type', 'lfc', 'abs_lfc',
        'pval', 'fdr', 'description', 'desc_gff', ]

#dflfc_noncod['geneid'] = dflfc_noncod['geneid'].astype(int)
pdwritecsv(dflfc_noncod, f'LNC_table_{mtd.case}.tsv', mtd.root_nc)
dflfc_noncod[cols].head()

In [ ]:
cols = ['LNC', 'protein_coding', 'hasSymbol', 'UNK']

dflfc_noncod = dflfc[ ~dflfc.biotype.isin(cols)]
fname = f'non_codings_degs_for_{mtd.case}.tsv'
# pdwritecsv(dflfc_noncod, fname, mtd.root_result, verbose=True)
print(len(dflfc_noncod), dflfc_noncod.biotype.unique())

dflfc_noncod.head()

In [ ]:
symbols = dflfc_noncod.symbol
len(symbols), ", ".join(symbols)

In [ ]:
dfin = dfdis[dfdis['ncRNA Symbol'].isin(symbols)]
print(len(dfin))
symb_identified_list = dfin['ncRNA Symbol'].unique()
len(symb_identified_list), symb_identified_list

In [ ]:
fname = f"lnc_disease_identification_for_{mtd.case}.tsv"
pdwritecsv(dfin, fname, mtd.root_nc, verbose=True)

In [ ]:
i = 0
print(len(dfin))
dfin[ dfin['ncRNA Symbol'] == symb_identified_list[i] ]

In [ ]:
dflfc_LFC = dflfc[ dflfc.biotype == 'LNC']

fname = f'LNC_special_non_codings_degs_for_{mtd.case}.tsv'
# pdwritecsv(dflfc_LFC, fname, mtd.root_result, verbose=True)
print(len(dflfc_LFC))
dflfc_LFC.head()

In [ ]:
symbols = dflfc_LFC.symbol.unique()
len(symbols), ", ".join(symbols)

In [ ]:
df_ident_LNC = dfdis[dfdis['ncRNA Symbol'].isin(symbols)]
symb_identified_list = df_ident_LNC['ncRNA Symbol'].unique()
symb_identified_list